LangChain + OpenAI API con Python

> Nota importante: en versiones antiguas se usaban clases como `LLMChain`, `SimpleSequentialChain` o memorias tipo `ConversationBufferMemory`.
> En LangChain moderno se recomienda trabajar con **LCEL / Runnables**, `ChatPromptTemplate`, parsers, retrievers y agentes creados con `create_agent`.

## 1. Setup

Instalar dependencias. Ejecutar una vez.

Paquetes principales:

- `langchain`: framework principal.
- `langchain-openai`: integración oficial con OpenAI.
- `langchain-core`: abstracciones base.
- `langchain-text-splitters`: división de textos para RAG.
- `pydantic`: validación de datos y salida estructurada.

Opcional: `python-dotenv` para cargar variables desde un archivo `.env`.

In [ ]:
%pip install --upgrade langchain langchain-openai langchain-core langchain-text-splitters pydantic python-dotenv

## 2. API Key y configuración

No hardcodear claves en el notebook. Usar variable de entorno.

Opciones:

1. Definir `OPENAI_API_KEY` en el entorno.
2. Crear un archivo `.env` con:
   ```text
   OPENAI_API_KEY=tu_api_key
   OPENAI_MODEL=gpt-5.4-nano-2026-03-17
   ```
3. Ingresarla manualmente cuando el notebook la pida.

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")

from openai import OpenAI

client = OpenAI()

print(f"Modelo: {OPENAI_MODEL}")
print("Cliente listo")

## 3. Primer contacto con LangChain


En la API directa de OpenAI solemos crear un cliente y llamar a un endpoint.
En LangChain usamos una abstracción de **chat model**.

La clase `ChatOpenAI` representa un modelo conversacional de OpenAI dentro del ecosistema LangChain.

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0.2,
)

response = model.invoke("Explicá qué es LangChain en una frase simple.")
print(response.content)

### 3.1 ¿Qué devuelve `invoke`?


Un modelo de chat devuelve un mensaje, no solo un string.

Por eso normalmente accedemos a:

```python
response.content
```

También existen metadatos útiles, por ejemplo uso de tokens, identificadores o información del proveedor.

In [ ]:
response.content

## 4. Mensajes: system, human y AI


Los modelos conversacionales trabajan con listas de mensajes.

Roles comunes:

- `system`: instrucciones generales, tono, reglas.
- `human`: mensaje del usuario.
- `ai`: mensaje previo del asistente.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="Sos un profesor de programación. Respondé claro y con ejemplos cortos."),
    HumanMessage(content="¿Para qué sirve una chain en LangChain?"),
]

response = model.invoke(messages)
print(response)

## 5. Prompt templates


Un **prompt template** permite reutilizar una estructura de prompt con variables.

Esto es clave en aplicaciones reales porque separa:

- plantilla del prompt,
- datos de entrada,
- modelo,
- salida.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un docente de IA para programadores. Explicá con precisión y sin humo."),
    ("human", "Explicá el concepto: {concepto}. Nivel: {nivel}. Incluí un ejemplo en Python."),
])

formatted_messages = prompt.invoke({
    "concepto": "prompt template",
    "nivel": "principiante",
})

formatted_messages

In [ ]:
response = model.invoke(formatted_messages)
print(response.content)

## 6. LCEL: LangChain Expression Language


La forma moderna de componer pasos en LangChain es usar **LCEL**.

Una chain básica puede escribirse así:

```python
chain = prompt | model | parser
```

El operador `|` conecta la salida de un paso con la entrada del siguiente.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt | model | parser

resultado = chain.invoke({
    "concepto": "LCEL",
    "nivel": "intermedio",
})

print(resultado)

## 7. Parámetros del modelo


Los parámetros se definen al crear el modelo

Parámetros útiles:

- `temperature`: controla variación/creatividad.
- `timeout`: tiempo máximo de espera.
- `max_retries`: reintentos ante errores transitorios.

In [ ]:
modelo_determinista = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0,
    max_retries=2,
    timeout=30,
)

modelo_creativo = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0.9,
    max_retries=2,
    timeout=30,
)

pregunta = "Dame 3 nombres para una app que resume PDFs con IA."

print("Determinista:")
print(modelo_determinista.invoke(pregunta).content)

print("\nCreativo:")
print(modelo_creativo.invoke(pregunta).content)

## 8. Output parsers


Un parser transforma la respuesta del modelo a un formato más cómodo.

El parser más simple es `StrOutputParser`, que extrae el texto final.

In [ ]:
prompt_resumen = ChatPromptTemplate.from_messages([
    ("system", "Resumí textos técnicos para estudiantes."),
    ("human", "Texto: {texto}\n\nDevolvé un resumen de 3 bullets."),
])

chain_resumen = prompt_resumen | model | StrOutputParser()

texto = '''
LangChain permite construir aplicaciones con modelos de lenguaje conectando prompts,
modelos, parsers, herramientas, retrievers y memoria. Su enfoque moderno se basa en
runnables y LCEL para componer flujos reutilizables.
'''

print(chain_resumen.invoke({"texto": texto}))

## 9. Salida estructurada con Pydantic


Cuando una aplicación necesita datos confiables, no alcanza con pedir “respondé en JSON”.
LangChain permite usar `with_structured_output(...)` para obtener objetos validados.

Ejemplo: extraer información de una descripción de curso.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Curso(BaseModel):
    titulo: str = Field(description="Título breve del curso")
    nivel: str = Field(description="Nivel: inicial, intermedio o avanzado")
    temas: List[str] = Field(description="Lista de temas principales")
    duracion_horas: float = Field(description="Duración estimada en horas")

structured_model = model.with_structured_output(Curso)

descripcion = '''
Curso intensivo para programadores que ya conocen Python y quieren aprender a crear
aplicaciones con la API de OpenAI, LangChain, RAG y agentes. Duración aproximada: 12 horas.
'''

curso = structured_model.invoke(
    f"Extraé los datos del curso desde esta descripción:\n{descripcion}"
)

curso.model_dump_json()

In [ ]:
print(curso.titulo)
print(curso.temas)
print(curso.model_dump())

## 10. Streaming


El streaming permite mostrar tokens a medida que se generan.
Es muy útil para interfaces de chat.

In [ ]:
prompt_stream = ChatPromptTemplate.from_messages([
    ("system", "Sos un asistente técnico. Respondé en español."),
    ("human", "Explicá en 5 bullets por qué LangChain es útil para RAG."),
])

chain_stream = prompt_stream | model | StrOutputParser()

for chunk in chain_stream.stream({}):
    print(chunk, end="", flush=True)

## 11. Batch


`batch` permite ejecutar la misma chain sobre muchas entradas.

Esto es útil para:

- clasificar muchos textos,
- resumir muchos documentos,
- generar evaluaciones,
- procesar filas de un dataset.

In [ ]:
prompt_clasificador = ChatPromptTemplate.from_messages([
    ("system", "Clasificá el sentimiento como Positivo, Neutral o Negativo. Respondé solo la etiqueta."),
    ("human", "Texto: {texto}"),
])

clasificador = prompt_clasificador | modelo_determinista | StrOutputParser()

entradas = [
    {"texto": "El curso fue claro y muy práctico."},
    {"texto": "No entendí nada y los ejemplos no funcionaban."},
    {"texto": "La clase empieza a las 18 hs."},
]

clasificador.batch(entradas)

## 12. Memoria conversacional


En tutoriales antiguos se usaba `ConversationBufferMemory`.
En LangChain moderno, una forma recomendada de manejar historial es envolver una chain con `RunnableWithMessageHistory`.

La idea:

1. Definir un prompt con un `MessagesPlaceholder`.
2. Crear una función que devuelva el historial según `session_id`.
3. Envolver la chain.
4. Invocar pasando `config={"configurable": {"session_id": "..."}}`.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate

prompt_chat = ChatPromptTemplate.from_messages([
    ("system", "Sos un tutor de Python y LangChain. Recordá el contexto de la conversación."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chat_chain = prompt_chat | model | StrOutputParser()

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chat_con_memoria = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "alumno-1"}}

print(chat_con_memoria.invoke(
    {"input": "Estoy aprendiendo LangChain y ya sé Python básico."},
    config=config,
))

print("\n--- segunda pregunta ---\n")

print(chat_con_memoria.invoke(
    {"input": "¿Qué debería aprender ahora? Respondé teniendo en cuenta lo que te dije antes."},
    config=config,
))

## 13. Tools y agentes


Un **tool** es una función que el modelo puede usar.

Un **agente** decide cuándo usar herramientas y cuándo responder directamente.

En LangChain moderno se usa `create_agent` desde `langchain.agents`.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def calcular_precio_con_iva(precio: float, iva: float = 0.21) -> float:
    '''Calcula el precio final sumando IVA. precio e iva deben ser números.'''
    return round(precio * (1 + iva), 2)

@tool
def recomendar_modelo(caso_de_uso: str) -> str:
    '''Recomienda una familia de solución según el caso de uso.'''
    caso = caso_de_uso.lower()
    if "rag" in caso or "document" in caso or "pdf" in caso:
        return "Usaría un modelo de chat + embeddings + retriever/vector store."
    if "clasificar" in caso or "sentimiento" in caso:
        return "Usaría un modelo de chat con salida estructurada y temperatura baja."
    return "Empezaría con un modelo de chat y evaluaría si hace falta RAG o tools."

agent = create_agent(
    model=model,
    tools=[calcular_precio_con_iva, recomendar_modelo],
    system_prompt="Sos un asistente para programadores. Usá tools cuando aporten precisión.",
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Tengo un producto de 100 USD sin IVA. ¿Cuánto sale con IVA de 21%?"}
    ]
})

result

In [ ]:
# Extraer el último mensaje del agente
ultimo_mensaje = result["messages"][-1]
print(ultimo_mensaje.content)